### LCEL 체인에 메모리 추가하기
- 질문할 때 과거 내용을 토대로 답변할 수 있는 체인을 만드는 작업

In [1]:
# ===== 패키지 설치 (최초 1회만 실행) =====
%pip install -q python-dotenv
%pip install -q -U langchain langchain-classic langchain-community langchain-openai langchain-teddynote networkx faiss-cpu

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
OpenAI 키: sk-proj-...
LANGSMITH 키: lsv2_pt_...
LangSmith 프로젝트: test0917
LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [2]:
from operator import itemgetter
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [4]:
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

C:\Users\user\AppData\Local\Temp\ipykernel_17420\2844409624.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")


In [5]:
memory.load_memory_variables({})  # 메모리 변수를 빈 딕셔너리로 초기화

{'chat_history': []}

In [6]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history") # memory_key 와 동일하게 입력
)

runnable.invoke({"input": "hi"})  # 메모리 로드 

{'input': 'hi', 'chat_history': []}

In [7]:
chat_history=RunnableLambda(memory.load_memory_variables) # assign() 메서드로 chat_history 변수에 할당

In [8]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    # | itemgetter("chat_history")  # memory_key 와 동일하게 입력
)

runnable.invoke({"input": "hi"})  # 메모리 로드

{'input': 'hi', 'chat_history': {'chat_history': []}}

In [9]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")
)

runnable.invoke({"input": "hi"})  # 메모리 로드

{'input': 'hi', 'chat_history': []}

In [10]:
chain = runnable | prompt | model

In [11]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [12]:
response = chain.invoke({"input": "만나서 반갑습니다. 제이름은 테디입니다. "})
print(response.content) # 생성된 응답을 출력

안녕하세요, 테디님! 만나서 저도 반가워요. 무엇을 도와드릴까요?


In [13]:
memory.load_memory_variables({})

{'chat_history': []}

In [14]:
memory.save_context( # 입력된 데이터와 응답 내용을 메모리에 저장
    {"human": "만나서 반갑습니다. 제이름은 테디입니다."}, {"ai": response.content}
)

memory.load_memory_variables({})  # 메모리 변수를 빈 딕셔너리로 초기화

{'chat_history': [HumanMessage(content='만나서 반갑습니다. 제이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요, 테디님! 만나서 저도 반가워요. 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [15]:
response = chain.invoke({"input": "제 이름이 무엇이었는지 기억하세요?"})
print(response.content) # 생성된 응답을 출력

네, 테디님이시죠. 어떤 도움이 필요하신가요?


### SQLite에 대화 내용 저장하기
- 인메모리 방식은 종료하면 대화 내용이 저장 x
- 이전 내용에서 이어나가면서 대화를 하고 싶다면 내화 내용을 데이터베이스에 기록

In [16]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from dotenv import load_dotenv

load_dotenv()

#SQLChatMessageHistory 객체를 생성하고 세션 ID와 데이터베이스 연결 파일을 설정
chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db" # 세션 ID와 데이터베이스 연결 파일을 설정
)

C:\Users\user\AppData\Local\Temp\ipykernel_17420\3119777446.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import SQLChatMessageHistory


In [17]:
# 사용자 메시지를 추가
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)
# AI 메시지를 추가
chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")

In [18]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [31]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."), # 시스템 메시지
        # 대화 기록을 위한 Placeholder
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"), # 질문
    ]
)

chain = prompt | ChatOpenAI(model_name="gpt-4o") | StrOutputParser()

In [32]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db",
    )

In [33]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

In [ ]:
# RunnableWithMessageHistory 객체를 생성하고, 대화 기록을 가져오는 함수와 입력 메시지 및 대화 기록의 키를 설정
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history, # 대화 기록을 가져오는 함수를 설정
    input_messages_key="question", # 입력 메시지의 키를 설정
    history_messages_key="chat_history", # 대화 기록의 키를 설정
    history_factory_config=config_fields, # 대화 기록 조회시 참고할 매개변수를 설정
)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [35]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [36]:
chain_with_history.invoke({"question":"안녕 반가워, 내 이름은 테디야"}, config)

'안녕하세요, 테디! 만나서 반가워요. 저는 당신을 도울 준비가 되어 있는 AI 어시스턴트입니다. 무엇을 도와드릴까요?'

In [37]:
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'당신의 이름은 테디라고 했습니다. 맞나요?'

In [39]:
#config 설정
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

# 질문과 config를 전달하여 실행
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'죄송하지만 저는 당신의 이름을 알 수 없습니다. 어떻게 도와드릴까요?'

### 휘발성 메모리로 일반 변수에 대화 내용 저장하기
- 세션이 종료되면 사라지는 휘발성 저장 방식

In [41]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("CH05-Memory") # 프로젝트명 입력
load_dotenv()  

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해 주세요.",
        ),
        # 대화 기록을 위한 key는 chat_history입니다. 이 부분은 변경하지 마세요!
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "#Question:\n{question}"), # 사용자 입력을 받는 부분
    ]
)

llm = ChatOpenAI(model_name="gpt-4o") # LLM 객체 생성

chain = prompt | llm | StrOutputParser() # 프롬프트와 LLM을 연결하여 체인 생성

LangSmith 추적을 시작합니다.
[프로젝트명]
CH05-Memory


In [42]:
store = {} # 세션 기록을 저장할 딕셔너리

# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store: # 세션 ID가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids] # 세션 ID에 해당하는 ChatMessageHistory 객체 반환

In [46]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history, # 세션 기록을 가져오는 함수
    input_messages_key="question", # 사용자의 질문이 템플릿 변수에 들어갈 키
    history_messages_key="chat_history", # 기록 메시지의 키
)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
chain_with_history.invoke(
    {"question": "나의 이름은 테디입니다. "}, # 질문 입력 
    config = {"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'안녕하세요, 테디님! 오늘 어떻게 도와드릴까요?'

In [48]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고?"}, # 질문입력
    config={"configurable": {"session_id": "abc1234"}}, # 세션 ID 기준으로 대화 기록을 가져오도록 설정
)

[대화 세션ID]: abc1234


'죄송하지만, 저는 당신의 이름을 알 수 없습니다.'